In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/moleculenet-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "feature-selection.py"

# 篩選親脂性模型的描述符
比較五套規則，用驗證集選出模型，再以保留的測試集評估。

第一次執行會由 DeepChem 下載 Lipophilicity。已有原始檔時可設定 DEEPCHEM_DATA_DIR。中間程式格寫入暫存腳本，小練習程式可接在腳本末端，再執行末兩格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

SEED = 42
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path.home() / ".cache" / "deepchem"))
cache.mkdir(parents=True, exist_ok=True)
tasks, (train, valid, test), _ = dc.molnet.load_lipo(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), save_dir=str(cache), reload=True)
assert tasks == ["exp"] and all(np.all(s.w[:, 0] > 0) for s in (train, valid, test))
print("train / valid / test:", len(train), len(valid), len(test))

def matrix(split, names):
    rows = []
    for smiles in split.ids:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"無法讀取分子 {smiles}")
        rows.append([getattr(Descriptors, name)(mol) for name in names])
    frame = pd.DataFrame(rows, columns=names)
    assert np.isfinite(frame.to_numpy()).all()
    return frame

def forest():
    return RandomForestRegressor(n_estimators=80, min_samples_leaf=2,
                                 random_state=SEED, n_jobs=1)

figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

continuous = ["MolWt", "HeavyAtomMolWt", "ExactMolWt", "MolLogP", "MolMR", "TPSA",
              "LabuteASA", "FractionCSP3", "BalabanJ", "BertzCT", "Chi0", "Chi0v",
              "Chi1", "Chi1v", "Kappa1", "Kappa2", "Kappa3"]
counts = ["HeavyAtomCount", "NumHAcceptors", "NumHDonors", "NumRotatableBonds",
          "RingCount", "NumAromaticRings", "NumRadicalElectrons"]
names = continuous + counts
X_train = matrix(train, names)
X_valid = matrix(valid, names)
y_train = train.y[:, 0]
y_valid = valid.y[:, 0]
variable = X_train.columns[VarianceThreshold().fit(X_train).get_support()].tolist()
corr = X_train[variable].corr().abs()
decorrelated = []
for name in variable:
    if all(corr.loc[name, other] <= 0.90 for other in decorrelated):
        decorrelated.append(name)

def rank_mi(columns):
    mi = mutual_info_regression(X_train[columns], y_train, random_state=SEED,
                                discrete_features=[name in counts for name in columns])
    return pd.Series(mi, index=columns).sort_values(ascending=False, kind="stable")

ranking = rank_mi(decorrelated)
sets = {"All": names, "Variable": variable, "Low correlation": decorrelated,
        "MI top 10": rank_mi(variable).head(10).index.tolist(),
        "Combined": ranking.head(10).index.tolist()}
scores, models = {}, {}
for label, columns in sets.items():
    models[label] = forest().fit(X_train[columns], y_train)
    scores[label] = mean_absolute_error(y_valid, models[label].predict(X_valid[columns]))
    print(f"{label:16} {len(columns):2}  valid MAE = {scores[label]:.4f}")
winner = min(scores, key=scores.get)
pred = models[winner].predict(matrix(test, names)[sets[winner]])
print(f"{winner}: test MAE = {mean_absolute_error(test.y[:, 0], pred):.4f}")
print("Constant:", sorted(set(names) - set(variable)))
print("MolWt / HeavyAtomMolWt:", corr.loc["MolWt", "HeavyAtomMolWt"])
print("Combined:", sets["Combined"])
fig, ax = plt.subplots(figsize=(7.2, 3.8), layout="constrained")
positions = np.arange(len(scores))
ax.barh(positions, list(scores.values()), color=["#236d75" if k == winner else "#89a9ae" for k in scores])
ax.set_yticks(positions, [f"{k} ({len(sets[k])})" for k in scores])
ax.invert_yaxis()
ax.set(xlabel="Validation MAE (logD units)", title="Predicting lipophilicity on unseen scaffolds")
ax.bar_label(ax.containers[0], fmt="%.3f", padding=4)
ax.set_xlim(0, max(scores.values()) * 1.15)
fig.savefig(figure_dir / "10-04-feature-selection.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "10-04-feature-selection.png")))